In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore')

# Lecture 11 #

### Rows from lists

In [ ]:
Table().with_columns('Numbers', make_array(1, 2, 3))

In [ ]:
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks

In [ ]:
drinks = drinks.with_rows([
    ['Matcha', 'Luminary', 5.75],
    ['Latte', 'Spyhouse',  6.40],
    ['Espresso',    'Spyhouse',  3.5],
    ['Espresso', "FSM",   2]
])
drinks

## Review: Group by one column

In [ ]:
movies = Table.read_table('top_movies_2017.csv')
movies.show(3)

In [ ]:
movies = movies.with_column('Gross (million $)', movies.column('Gross (Adjusted)') / 1_000_000)
movies.show(3)

In [ ]:
movies.group('Studio')

In [ ]:
# selecting only the columns we care about and sorting by gross - more readable
(movies
    .select('Studio', 'Gross (million $)')
    .group('Studio', np.average)
    .sort('Gross (million $) average', descending=True)
    .show(23))

## Cross-classification: grouping by two columns

In [ ]:
# grouping by two variables can be useful but also overwhelming -
# there are way too many options for Studio/Year combinations to make this readable or useful
movies.group(['Studio', 'Year']).show(3)

In [ ]:
# idea: create a new column 'Decade' that is the decade of the movie's release
# we might get something more readable if we group by decade instead of year
movies = movies.with_column('Decade', (movies.column('Year') // 10) * 10)
movies.show(3)

In [ ]:
movies.group(['Studio', 'Decade']).show(3)

In [ ]:
# see trends over time for a few studios!
movies.group(['Studio', 'Decade'],
             np.average).show(5)

In [ ]:
# select only the columns we care about for a less cluttered output
movies.select('Studio', 'Decade', 'Gross (million $)').group(['Studio', 'Decade'], np.average).show(10)

## Pivot Tables

In [ ]:
# the pivot function does the same thing as grouping with two columns
# but it displays the results in a different layout
movies.pivot('Decade', 'Studio') # default is count, just like with .group()

In [ ]:
# can also specify a different aggregation function, just like with .group()
movies.pivot('Decade', 
             'Studio', 
             values='Gross (million $)',
             collect=np.average)

## Practice Problems ##

In [ ]:
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky = Table.read_table('skyscrapers.csv')
sky = (sky.with_column('age', 2025 - sky.column('completed'))
          .drop('completed'))
sky.show(3)

In [ ]:
# 1. For each city, what’s the tallest building for each material?

sky.select('material', 'city', 'height').group(['city', 'material'], collect=max)

In [ ]:
# 2. For each city, what’s the height difference between the tallest 
#    steel building and the tallest concrete building?

In [ ]:
# hint: try a pivot table first! where would you go from here?
sky_p = sky.pivot('material', 'city', values='height', collect=max)
sky_p.show()

In [ ]:
sky_p.column('steel') - sky_p.column('concrete')

In [ ]:
sky_p = sky_p.with_column('difference (steel - concrete)',
                            sky_p.column('steel') - sky_p.column('concrete'))
sky_p.show()
# hmm i'd prefer not to have negative differences

In [ ]:
# try again using absolute difference
sky_p = sky_p.with_column('difference (steel - concrete)',
                            abs(sky_p.column('steel') - sky_p.column('concrete')))
sky_p.show()

In [ ]:
sky_p.sort('difference (steel - concrete)', descending=True)

In [ ]:
# 3. Generate a table of the names of the oldest buildings for each 
#    material for each city:

# Hint: You can use sort to find the name of the oldest building in the dataset
sky.sort('age', descending=True).column('name').item(0)

In [ ]:
def first(s):
    "Return the first element in an array."
    return s.item(0)

names_of_oldest_buildings = sky.sort('age', descending=True).pivot('material', 'city', 'name', first)
names_of_oldest_buildings

In [ ]:
names_of_oldest_buildings.where('city', 'Minneapolis')

## Joins ##

In [ ]:
drinks

In [ ]:
# replicating the table from the slides
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Luminary', 'Spyhouse', 'Luminary')
)
discounts

In [ ]:
# joining on a common column - the column label does NOT have to be the same, but two columns should share values
# in this case, the shared values are 'Luminary' and 'Spyhouse' 
# (not 'JS Bean'??)
combined = drinks.join('Cafe', discounts, 'Location')
combined

In [ ]:
# now we can use the information from the discount table and the information from the drinks table
# to compute the discounted price of each drink at each cafe
# option 1: array arithmetic and .with_column()
discounted_frac = 1 - combined.column('Coupon % off') / 100
combined.with_column(
    'Discounted Price', 
    combined.column('Price') * discounted_frac
)

In [ ]:
# option 2 - use .apply() with a function on a ROW (not a column, like we've done before)
# rows are not arrays - why? because they can contain different types of data
# but we can still access elements of the row using .item(index), similar to arrays
def get_discounted_price(row):
    discounted_frac = 1 - row.item(3) / 100 # index 3 is the Coupon % off column
    return row.item(2) * discounted_frac # index 2 is the Price column

combined.with_column(
    'Discounted Price',
    combined.apply(get_discounted_price)
)

In [ ]:
# what happens if we join a column to itself?
drinks.join('Cafe', drinks, 'Cafe')

In [ ]:
# we get all combinations of drinks at each cafe.
# this table could be useful if I wanted to know, for example, all the possible pairs of drinks I could order at each cafe.